# Cross-Asset OFI Reproduction Notebook

The notebook is the executable reproduction record for the project. It follows the methodology in `report/methodology.md` and runs the project stages in the same order used to produce the committed `output/` artifacts.

The full run is long because it reconstructs order books when needed, runs the 60-second replication, broad search grid, candidate confirmation, robustness tests, sub-minute grids, decay curve, publication-figure generation, and temporal-stability check.

## 0. Environment Setup

The notebook is intended to be run from either the repository root or the `notebooks/` folder. All commands are executed from the repository root so relative paths match the scripts.

In [ ]:
from pathlib import Path
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
RAW = DATA / 'historical'
PROCESSED = DATA / 'processed'
OUTPUT = ROOT / 'output'

COMMAND_LOG = []

def run_step(args, label):
    start = time.time()
    cmd = [sys.executable, *args]
    print(f'[{label}] ' + ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)
    elapsed = time.time() - start
    COMMAND_LOG.append({'stage': label, 'command': ' '.join(cmd), 'seconds': round(elapsed, 1)})
    print(f'[{label}] completed in {elapsed:.1f}s')

def file_table(path):
    rows = []
    for p in sorted(path.rglob('*')):
        if p.is_file():
            rows.append({'file': str(p.relative_to(ROOT)).replace('\\\\', '/'), 'bytes': p.stat().st_size})
    return pd.DataFrame(rows)

ROOT

## 1. Install Python Dependencies

`requirements.txt` contains the libraries used by the scripts. This cell is included so a fresh environment can run the notebook end to end.

In [ ]:
run_step(['-m', 'pip', 'install', '-r', 'requirements.txt'], 'install_requirements')

## 2. Data Inventory and Source

Raw data are not included in the GitHub repository. A reproducer must download the OKX L2 snapshot/update `.data` files separately and place them under `data/historical/` before running the third code cell below. The expected sample is BTC-USDT, ETH-USDT, SOL-USDT and XRP-USDT from 2026-05-02 to 2026-06-26, with filenames matching `<ASSET>-USDT-L2orderbook-<depth>lv-YYYY-MM-DD.data`.

The third code cell inventories the local raw files. Run it only after the raw data have been placed in `data/historical/`. Raw and processed data are excluded from Git because they are large local artifacts.

In [ ]:
raw_files = sorted(RAW.glob('*.data'))
processed_files = sorted(PROCESSED.glob('*.npz'))
if not raw_files:
    raise FileNotFoundError(
        'No raw OKX .data files found in data/historical/. Download the raw L2 files '
        'for BTC-USDT, ETH-USDT, SOL-USDT and XRP-USDT covering 2026-05-02 '
        'to 2026-06-26, then rerun this third code cell.'
    )

inventory = pd.DataFrame({
    'location': ['data/historical', 'data/processed'],
    'file_count': [len(raw_files), len(processed_files)],
    'first_file': [raw_files[0].name if raw_files else '', processed_files[0].name if processed_files else ''],
    'last_file': [raw_files[-1].name if raw_files else '', processed_files[-1].name if processed_files else ''],
})
inventory

## 2.1 Modelling Definitions Used By The Code

The scripts below implement the formal definitions in `report/methodology.md`. The key objects are:

```text
OFI_i,m,n      = bid_flow_i,m,n - ask_flow_i,m,n
OFI_i,m,k      = sum of OFI_i,m,n over snapshot transitions inside bar k
x_i,k          = scalar OFI feature after depth aggregation and normalisation
z_i,k^(ell)    = sum_{u=0}^{ell-1} x_i,k-u
y_i,k^(h)      = sum_{u=1}^{h} r_i,k+u
PI/FPI model   = target return predicted from the target asset's own OFI
CI/FCI model   = target return predicted from all assets' OFI
placebo model  = cross-asset OFI replaced by exact one-calendar-day shifted OFI
```

The execution cells preserve this ordering: reconstruct books, build OFI, align panels, create lag-window features, fit walk-forward models, compare against controls, apply multiple-comparison corrections, then generate reports and figures.

## 3. Reconstruct Daily Order-Book Caches

`scripts/reconstruct.py` converts raw OKX `.data` files into compressed daily `.npz` files. Existing processed files are skipped by the script, so the cell can be rerun without overwriting completed caches.

In [ ]:
run_step([
    'scripts/reconstruct.py',
    '--raw-dir', 'data/historical',
    '--out-dir', 'data/processed',
    '--levels', '20',
    '--jobs', '1',
    '--assets', 'btc', 'eth', 'sol', 'xrp',
    '--start', '2026-05-02',
    '--end', '2026-06-26',
], 'reconstruct_books')

processed_files = sorted(PROCESSED.glob('*.npz'))
pd.DataFrame({'processed_file': [p.name for p in processed_files[:10]], 'bytes': [p.stat().st_size for p in processed_files[:10]]})

## 4. Focused 60-Second Evidence Pass

`scripts/core_results.py` builds the 60-second CCZ-style panel and writes the focused construction, contemporaneous, predictive, lead-lag, extension and portfolio outputs to `output/core/`.

In [ ]:
run_step(['scripts/core_results.py'], 'core_results')
file_table(OUTPUT / 'core')

In [ ]:
core_predictive = pd.read_csv(OUTPUT / 'core' / 'predictive.csv')
core_predictive.sort_values('cross_minus_own_r2', ascending=False).head(10)

The focused pass checks whether the direct 60-second paper-style specification is enough. The later stages are required because this direct predictive replication is not the final supported result.

## 5. Broad Predictive Grid

`src/tests.py` runs the broad discovery grid across bar widths, OFI schemes, lag sets and forecast horizons. It writes incrementally to `output/grid/results.csv`, so interrupted runs can resume.

In [ ]:
run_step(['src/tests.py'], 'broad_grid')
grid_results = pd.read_csv(OUTPUT / 'grid' / 'results.csv')
grid_results['status'].value_counts(dropna=False)

## 6. Summarise and Rank the Broad Grid

`scripts/analyze_tests_results.py` reads `output/grid/results.csv` and produces the ranked search artifacts in `output/grid/`.

In [ ]:
run_step(['scripts/analyze_tests_results.py'], 'summarise_grid')
file_table(OUTPUT / 'grid')

In [ ]:
ranked = pd.read_csv(OUTPUT / 'grid' / 'ranked.csv')
ranked[['key', 'bar_s', 'scheme', 'lagset', 'horizon', 'cross_minus_own_r2', 'cross_minus_car_r2', 'pnl_net']].head(15)

The broad grid is discovery only. It identifies candidate families, but the final interpretation depends on confirmation, placebo checks and multiple-comparison correction.

## 7. Candidate Confirmation

`scripts/confirm_candidates.py` reruns the strongest broad-grid candidate families with denser refits, two training windows and top-10/top-20 depth variants. Outputs are written to `output/confirmation/`.

In [ ]:
run_step(['scripts/confirm_candidates.py'], 'confirm_candidates')
confirmation = pd.read_csv(OUTPUT / 'confirmation' / 'confirmation.csv')
confirmation[['tag', 'bar_s', 'scheme', 'lagset', 'horizon', 'train', 'levels', 'cross_minus_own_r2', 'cross_minus_car_r2', 'pnl_net']].head(15)

## 8. Candidate Significance and Portfolio Checks

`scripts/candidate_significance.py` applies DM/MCS tests and cost-aware forecast-implied portfolio diagnostics to the 300s PCA CCZ-lag family.

In [ ]:
run_step(['scripts/candidate_significance.py'], 'candidate_significance')
file_table(OUTPUT / 'confirmation')

## 9. 300s CCZ Robustness Battery

`scripts/robustness_300s_ccz.py` evaluates the 300s candidate against day-shifted placebo, cross-return history and full-grid corrected significance. This stage determines whether the slower candidate is a supported finding or only exploratory.

In [ ]:
run_step(['scripts/robustness_300s_ccz.py'], 'robustness_300s')
robust_300s = pd.read_csv(OUTPUT / 'robustness_300s' / 'significance.csv')
pd.DataFrame({
    'test': ['cross beats own', 'cross beats cross returns', 'real beats placebo'],
    'passing_cells': [
        int((robust_300s['p_own_vs_cross_bonf672'] < 0.05).sum()),
        int((robust_300s['p_car_vs_cross_bonf672'] < 0.05).sum()),
        int((robust_300s['p_placebo_vs_real_cross_bonf672'] < 0.05).sum()),
    ],
    'total_cells': len(robust_300s),
})

The 300s candidate fails the cross-return and placebo controls after correction, so it is not the headline finding.

## 10. Sub-Minute 1s/5s/10s Grid

`src/tests.py --spec subminute` builds the common-second panel, aggregates it to 1s, 5s and 10s bars, and evaluates the fixed-UTC tuned-ridge sub-minute model. Headline outputs are written to `output/subminute/results.csv`.

In [ ]:
run_step(['src/tests.py', '--spec', 'subminute'], 'subminute_grid')
subminute_results = pd.read_csv(OUTPUT / 'subminute' / 'results.csv')
subminute_results[['bar_s', 'scheme', 'cross_minus_own_r2', 'cross_minus_car_r2', 'cross_hit', 'cross_unit_pnl_bps', 'status']]

## 11. Sub-Minute Robustness and Corrected Significance

`scripts/subminute_robustness.py` reruns the paired real/placebo evaluator, writes `output/subminute/placebo.csv` and `output/subminute/significance.csv`, and writes the summary report in `output/subminute/report.md`.

In [ ]:
run_step(['scripts/subminute_robustness.py'], 'subminute_robustness')
sub_sig = pd.read_csv(OUTPUT / 'subminute' / 'significance.csv')
pd.DataFrame({
    'test': ['cross beats own', 'cross beats cross returns', 'real beats placebo'],
    'passing_cells': [
        int((sub_sig['p_own_vs_cross_bonf684'] < 0.05).sum()),
        int((sub_sig['p_car_vs_cross_bonf684'] < 0.05).sum()),
        int((sub_sig['p_placebo_vs_real_cross_bonf684'] < 0.05).sum()),
    ],
    'total_cells': len(sub_sig),
})

The sub-minute grid is the strongest positive result: all 1s and 5s cells pass the corrected own, cross-return and placebo controls.

## 12. Sub-Minute 1s-30s Decay Grid

`scripts/subminute_decay.py` extends the sub-minute design to every integer bar width from 1s to 30s. It writes `results.csv`, `placebo.csv`, `significance.csv` and `report.md` under `output/subminute_decay/`.

In [ ]:
run_step(['scripts/subminute_decay.py'], 'subminute_decay')
decay_sig = pd.read_csv(OUTPUT / 'subminute_decay' / 'significance.csv')
decay = decay_sig.assign(
    pass_own=decay_sig['p_own_vs_cross_bonf792'] < 0.05,
    pass_car=decay_sig['p_car_vs_cross_bonf792'] < 0.05,
    pass_placebo=decay_sig['p_placebo_vs_real_cross_bonf792'] < 0.05,
)
decay['pass_all'] = decay[['pass_own', 'pass_car', 'pass_placebo']].all(axis=1)
decay.groupby('bar_s')[['pass_own', 'pass_car', 'pass_placebo', 'pass_all']].sum().head(12)

## 13. Generate the Decay-Curve Graph

`scripts/plot_subminute_decay.py` reads `output/subminute_decay/results.csv` and `output/subminute_decay/significance.csv`, writes the plotted data to `curve_data.csv`, and saves the graph as `curve.png`.

In [ ]:
run_step(['scripts/plot_subminute_decay.py'], 'plot_subminute_decay')
curve_data = pd.read_csv(OUTPUT / 'subminute_decay' / 'curve_data.csv')
curve_data[['bar_s', 'scheme', 'cross_minus_car_r2', 'pass_all_bonf792']].head(12)

In [ ]:
display(Image(filename=str(OUTPUT / 'subminute_decay' / 'curve.png')))

The graph is therefore produced directly from the same decay-grid CSVs used for the report, not from a manually edited plotting file.

## 14. Generate Publication Figures

`scripts/plot_publication_figures.py` turns the committed output CSVs into the visual summaries used in the README, report and methodology.

In [ ]:
run_step(['scripts/plot_publication_figures.py'], 'publication_figures')

In [ ]:
display(Image(filename=str(OUTPUT / 'figures' / '01_predictive_power_vs_baselines.png')))
display(Image(filename=str(OUTPUT / 'figures' / '02_cross_edge_decay.png')))
display(Image(filename=str(OUTPUT / 'figures' / '03_corrected_pass_heatmap.png')))

## 15. Temporal-Stability Check

`scripts/subminute_temporal_stability.py` tests the already-selected 1s-6s region on one fixed calendar split inside the same sample: first 6 weeks for training and final 2 weeks for evaluation. This is not a fresh holdout.

In [ ]:
run_step(['scripts/subminute_temporal_stability.py'], 'subminute_temporal_stability')
stability = pd.read_csv(OUTPUT / 'subminute_decay' / 'temporal_stability.csv')
pd.DataFrame({
    'passing_all_three_raw_controls': [int(stability['all_three_controls_raw_5pct'].sum())],
    'total_cells': [len(stability)],
})

## 16. Output Provenance Manifest

The table below records which notebook stage produces each output family. Running all cells above regenerates the same project output structure.

In [ ]:
provenance = pd.DataFrame([
    {'output': 'data/processed/*.npz', 'producer': 'scripts/reconstruct.py'},
    {'output': 'output/core/*', 'producer': 'scripts/core_results.py'},
    {'output': 'output/grid/results.csv', 'producer': 'src/tests.py'},
    {'output': 'output/grid/summary.md, ranked.csv, asset_ranked.csv', 'producer': 'scripts/analyze_tests_results.py'},
    {'output': 'output/confirmation/confirmation.*', 'producer': 'scripts/confirm_candidates.py'},
    {'output': 'output/confirmation/significance*, portfolio.csv', 'producer': 'scripts/candidate_significance.py'},
    {'output': 'output/robustness_300s/*', 'producer': 'scripts/robustness_300s_ccz.py'},
    {'output': 'output/subminute/results.csv', 'producer': 'src/tests.py --spec subminute'},
    {'output': 'output/subminute/placebo.csv, significance.csv, report.md', 'producer': 'scripts/subminute_robustness.py'},
    {'output': 'output/subminute_decay/results.csv, placebo.csv, significance.csv, report.md', 'producer': 'scripts/subminute_decay.py'},
    {'output': 'output/subminute_decay/curve_data.csv, curve.png', 'producer': 'scripts/plot_subminute_decay.py'},
    {'output': 'output/figures/*', 'producer': 'scripts/plot_publication_figures.py'},
    {'output': 'output/subminute_decay/temporal_stability.csv, temporal_stability_report.md', 'producer': 'scripts/subminute_temporal_stability.py'},
])
provenance

In [ ]:
all_outputs = file_table(OUTPUT)
all_outputs

## 17. Command Log for the Notebook Run

After a full execution, this table records the actual commands executed by the notebook and the elapsed time for each stage.

In [ ]:
pd.DataFrame(COMMAND_LOG)